In [18]:
# Project by Mahdi Islam
# Started 5/5/2026
# Goal is to create a machine learning algorithm to learn from listening habits to give better reccomendations

In [19]:
from tensorflow import keras
from keras import models, layers
import tensorflow as tf
from datetime import datetime
import pandas as pd
import numpy as np
keras.utils.set_random_seed(552026)

In [20]:
# The following parameters are the inputs for the model
# track_id -- Embedded
# artist_id -- Embedded
# hour_sin
# hour_cos
# day_sin
# day_cos
# target_score


In [21]:
train = "model_ready_history_train.csv" # input file for bulk training
validation = "model_ready_history_val.csv" # validation file to refine training
test = "model_ready_history_test.csv" # test file to check if its good

# Load the data files
train_df = pd.read_csv(train)
val_df = pd.read_csv(validation)
test_df = pd.read_csv(test)

# Check that the files loaded correctly
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

train_df.head()

(14859, 7)
(1857, 7)
(1858, 7)


,track_id,artist_id,hour_sin,hour_cos,day_sin,day_cos,target_score
0,Mob Ties,Drake,-0.500000,8.660254e-01,0.000000,1.00000,0.50
1,Upset (feat. Tommy Richman & FELIX!),Brent Faiyaz,-0.707107,-7.071068e-01,-0.781831,0.62349,0.80
2,Bags,Clairo,1.000000,6.123234e-17,0.781831,0.62349,0.20
3,ball w/o you,21 Savage,-0.258819,9.659258e-01,0.781831,0.62349,0.00
4,ball w/o you,21 Savage,-0.258819,9.659258e-01,0.781831,0.62349,0.25


In [22]:
if 'input' in globals():
    del input
    print("Deleted 'input' variable to prevent conflict with built-in function.")

In [23]:
track_column = "track_id"
artist_column = "artist_id"

numeric_columns = [
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]

model_features = [
    track_column,
    artist_column,
] + numeric_columns

target_column = "target_score"

X_train = train_df[model_features]
y_train = train_df[target_column]

X_val = val_df[model_features]
y_val = val_df[target_column]

X_test = test_df[model_features]
y_test = test_df[target_column]

print(X_train.shape, y_train.shape) # check the shape of the data

(14859, 6) (14859,)


In [24]:
# Helper function to aid in embedding for tracks and artists
track_lookup = tf.keras.layers.StringLookup(
    vocabulary=X_train["track_id"].unique(),
    mask_token=None,
    num_oov_indices=1
)

artist_lookup = tf.keras.layers.StringLookup(
    vocabulary=X_train["artist_id"].unique(),
    mask_token=None,
    num_oov_indices=1
)

In [25]:
track_embedding_dim = 32
artist_embedding_dim = 16

# Track embedding creation
track_embedding = tf.keras.layers.Embedding(
    input_dim=track_lookup.vocabulary_size(),
    output_dim=track_embedding_dim
)
track_input = tf.keras.Input(shape=(1,), name="track_id", dtype=tf.string)
track_index = track_lookup(track_input)
track_vector = track_embedding(track_index)
track_vector = tf.keras.layers.Flatten()(track_vector)

# Artist embedding creation
artist_embedding = tf.keras.layers.Embedding(
    input_dim=artist_lookup.vocabulary_size(),
    output_dim=artist_embedding_dim
)
artist_input = tf.keras.Input(shape=(1,), name="artist_id", dtype=tf.string)
artist_index = artist_lookup(artist_input)
artist_vector = artist_embedding(artist_index)
artist_vector = tf.keras.layers.Flatten()(artist_vector)

In [26]:
# Creating inputs for the model
hour_sin_input = tf.keras.Input(shape=(1,), name="hour_sin", dtype=tf.float32)
hour_cos_input = tf.keras.Input(shape=(1,), name="hour_cos", dtype=tf.float32)
day_sin_input = tf.keras.Input(shape=(1,), name="day_sin", dtype=tf.float32)
day_cos_input = tf.keras.Input(shape=(1,), name="day_cos", dtype=tf.float32)

In [27]:
# Concatenating all inputs in one layer
full_input_layer = tf.keras.layers.Concatenate()([
    track_vector,
    artist_vector,
    hour_sin_input,
    hour_cos_input,
    day_sin_input,
    day_cos_input,
])
combined_size = track_embedding_dim + artist_embedding_dim + len(numeric_columns)

In [28]:
# Creating the model

# Creating the prediction part of the model
prediction_head = models.Sequential([
    # Input layer takes in full_input_layer, with embeddings and numerical inputs
    layers.InputLayer(input_shape=(52,)),
    # First Dense neuron layer with 128 neurons
    layers.Dense(128, activation="relu"),
    # Second Dense neuron layer with 68 neurons
    layers.Dense(64, activation="relu"),
    # Output layer
    layers.Dense(1, activation="sigmoid")
])

# Connecting the input, prediction, and output
output = prediction_head(full_input_layer)

# Defining the model
model = models.Model(
    inputs=[
        track_input,
        artist_input,
        hour_sin_input,
        hour_cos_input,
        day_sin_input,
        day_cos_input
        ],
    outputs=output
)

# Compiling the full model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ track_id            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ artist_id           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ string_lookup_2     │ (None, 1)         │          0 │ track_id[0][0]    │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ string_lookup_3     │ (None, 1)         │          0 │ artist_id[0][0]   │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1, 32)     │     57,248 │ string_lookup_2[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 1, 16)     │      7,456 │ string_lookup_3[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 32)        │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 16)        │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hour_sin            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hour_cos            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_sin             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_cos             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 52)        │          0 │ flatten_2[0][0],  │
│ (Concatenate)       │                   │            │ flatten_3[0][0],  │
│                     │                   │            │ hour_sin[0][0],   │
│                     │                   │            │ hour_cos[0][0],   │
│                     │                   │            │ day_sin[0][0],    │
│                     │                   │            │ day_cos[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 1)         │     15,105 │ concatenate_1[0]… │
│ (Sequential)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 79,809 (311.75 KB)

 Trainable params: 79,809 (311.75 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
# Preparing to train the model

# Creating input dictionaries for model training
X_train_dict = {
    "track_id": tf.constant(X_train["track_id"], dtype=tf.string),
    "artist_id": tf.constant(X_train["artist_id"], dtype=tf.string),
    "hour_sin": tf.constant(X_train["hour_sin"], dtype=tf.float32),
    "hour_cos": tf.constant(X_train["hour_cos"], dtype=tf.float32),
    "day_sin": tf.constant(X_train["day_sin"], dtype=tf.float32),
    "day_cos": tf.constant(X_train["day_cos"], dtype=tf.float32),
}

X_val_dict = {
    "track_id": tf.constant(X_val["track_id"], dtype=tf.string),
    "artist_id": tf.constant(X_val["artist_id"], dtype=tf.string),
    "hour_sin": tf.constant(X_val["hour_sin"], dtype=tf.float32),
    "hour_cos": tf.constant(X_val["hour_cos"], dtype=tf.float32),
    "day_sin": tf.constant(X_val["day_sin"], dtype=tf.float32),
    "day_cos": tf.constant(X_val["day_cos"], dtype=tf.float32),
}

X_test_dict = {
    "track_id": tf.constant(X_test["track_id"], dtype=tf.string),
    "artist_id": tf.constant(X_test["artist_id"], dtype=tf.string),
    "hour_sin": tf.constant(X_test["hour_sin"], dtype=tf.float32),
    "hour_cos": tf.constant(X_test["hour_cos"], dtype=tf.float32),
    "day_sin": tf.constant(X_test["day_sin"], dtype=tf.float32),
    "day_cos": tf.constant(X_test["day_cos"], dtype=tf.float32),
}

# Training the model
predictor = model.fit(
    X_train_dict,
    y_train,
    validation_data=(X_val_dict, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.0776 - mae: 0.2083 - val_loss: 0.0558 - val_mae: 0.1582
Epoch 2/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0654 - mae: 0.1784 - val_loss: 0.0561 - val_mae: 0.1520
Epoch 3/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0601 - mae: 0.1668 - val_loss: 0.0565 - val_mae: 0.1503
Epoch 4/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0562 - mae: 0.1589 - val_loss: 0.0578 - val_mae: 0.1502
Epoch 5/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0532 - mae: 0.1512 - val_loss: 0.0589 - val_mae: 0.1518
Epoch 6/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0508 - mae: 0.1446 - val_loss: 0.0604 - val_mae: 0.1527
Epoch 7/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0490 - mae: 0.1389 - val_loss: 0.0616 - val_mae: 0.1534
Epoch 8/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0474 - mae: 0.1342 - val_loss: 0.0623 - val_mae: 0.1541
Epoch 9/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - lo

In [30]:
test_loss, test_mae = model.evaluate(X_test_dict, y_test)

print("Test loss:", test_loss)
print("Test MAE:", test_mae)

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0802 - mae: 0.1627
Test loss: 0.08021160960197449
Test MAE: 0.1626950055360794


In [32]:
# Generate predictions on the test set
predictions = model.predict(X_test_dict, verbose=0).flatten()

# Create a results table from the test input data
test_results = X_test.copy().reset_index(drop=True)

# Add predicted score, actual score, and error
test_results["predicted_score"] = predictions
test_results["actual_score"] = y_test.reset_index(drop=True)
test_results["error"] = abs(test_results["actual_score"] - test_results["predicted_score"])

# Put index into its own column
clean_predictions = test_results.reset_index()

# Rename columns for readability
clean_predictions = clean_predictions.rename(columns={
    "index": "row",
    "artist_id": "artist",
    "predicted_score": "predicted",
    "actual_score": "actual"
})

# Round numerical columns
round_columns = [
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "predicted",
    "actual",
    "error"
]

clean_predictions[round_columns] = clean_predictions[round_columns].round(3)

# Reorder columns
clean_predictions = clean_predictions[
    [
        "row",
        "track_id",
        "artist",
        "hour_sin",
        "hour_cos",
        "day_sin",
        "day_cos",
        "predicted",
        "actual",
        "error",
    ]
]

# Display table
clean_predictions.head(25)

,row,track_id,artist,hour_sin,hour_cos,day_sin,day_cos,predicted,actual,error
0,0,Change,J. Cole,0.259,-0.966,0.975,-0.223,0.998,0.50,0.498
1,1,Change,J. Cole,0.259,-0.966,0.975,-0.223,0.998,0.65,0.348
2,2,Baby Keem,Baby Keem,0.000,-1.000,0.975,-0.223,0.979,1.00,0.021
3,3,True Love,Kanye West,0.000,-1.000,0.975,-0.223,0.955,1.00,0.045
4,4,"GONE, GONE / THANK YOU","Tyler, The Creator",0.000,-1.000,0.975,-0.223,0.938,1.00,0.062
5,5,November Has Come,Gorillaz,0.000,-1.000,0.975,-0.223,0.937,1.00,0.063
6,6,Bad Boy (with Young Thug),Juice WRLD,0.000,-1.000,0.975,-0.223,0.846,1.00,0.154
7,7,Super Rich Kids,Frank Ocean,0.000,-1.000,0.975,-0.223,0.941,1.00,0.059
8,8,Champion,Kanye West,0.000,-1.000,0.975,-0.223,0.930,1.00,0.070
9,9,Ride Wit Me,Nelly,0.000,-1.000,0.975,-0.223,0.815,1.00,0.185


In [33]:
# Helper function to convert hour into sine/cosine features
def encode_hour(hour):
    hour = int(hour)
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    return hour_sin, hour_cos

# Helper function to convert day of week into sine/cosine features
# Monday = 0, Tuesday = 1, ..., Sunday = 6
def encode_day(day_of_week):
    day_of_week = int(day_of_week)
    day_sin = np.sin(2 * np.pi * day_of_week / 7)
    day_cos = np.cos(2 * np.pi * day_of_week / 7)
    return day_sin, day_cos

# Ask user for input
track_id = input("Enter track_id, example spotify:track:... : ")
artist_id = input("Enter artist_id / artist name: ")

hour = input("Enter hour of day, 0-23: ")
day_of_week = input("Enter day of week, Monday=0, Tuesday=1, ..., Sunday=6: ")

# Encode time features
hour_sin, hour_cos = encode_hour(hour)
day_sin, day_cos = encode_day(day_of_week)

# Create model input dictionary, ensuring correct TensorFlow dtypes
single_input = {
    "track_id": tf.constant([track_id], dtype=tf.string),
    "artist_id": tf.constant([artist_id], dtype=tf.string),
    "hour_sin": tf.constant([hour_sin], dtype=tf.float32),
    "hour_cos": tf.constant([hour_cos], dtype=tf.float32),
    "day_sin": tf.constant([day_sin], dtype=tf.float32),
    "day_cos": tf.constant([day_cos], dtype=tf.float32),
}

# Predict score
predicted_score = model.predict(single_input, verbose=0)[0][0]

print("Predicted target score:", round(float(predicted_score), 3))

Enter track_id, example spotify:track:... : Bandit
Enter artist_id / artist name: Juice Wrld
Enter hour of day, 0-23: 3
Enter day of week, Monday=0, Tuesday=1, ..., Sunday=6: 5
Predicted target score: 0.834
